# All-Model Full Metrics Evaluation (Fresh Inference)

This notebook computes metrics from **all available model checkpoints** and builds a single comparison table.

## Computed metrics
- Classification: AUROC, AUPRC, Accuracy, Precision, Recall, Specificity, F1, FPR, FNR
- Reconstruction: MSE, MAE, SSIM (overall + healthy/lesion splits)
- Lesion localization (if masks exist): Dice, IoU

## Outputs
- `evaluations/tables/all_models_full_metrics.csv`
- `evaluations/tables/all_models_full_metrics_ranked.csv`
- `evaluations/tables/all_models_full_metrics_publication.csv`

In [ ]:
# Environment and imports (GitHub-first in Colab)
import os
import sys
import importlib.util
import subprocess
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

# Detect Colab
try:
    from google.colab import drive
    IN_COLAB = True
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except Exception:
    IN_COLAB = False

# GitHub-first repo resolution
REPO_URL = 'https://github.com/RifaDeen/symAD-ECNN.git'
COLAB_REPO = Path('/content/symAD-ECNN')

if IN_COLAB:
    if not COLAB_REPO.exists():
        print('Cloning repository from GitHub...')
        subprocess.check_call(['git', 'clone', REPO_URL, str(COLAB_REPO)])
    else:
        print('Repository exists. Pulling latest changes...')
        subprocess.check_call(['git', '-C', str(COLAB_REPO), 'pull'])
    REPO_ROOT = COLAB_REPO
else:
    if Path('C:/Users/rifad/symAD-ECNN').exists():
        REPO_ROOT = Path('C:/Users/rifad/symAD-ECNN')
    else:
        REPO_ROOT = Path.cwd().resolve()

# Project root (Drive preferred when available)
if Path('/content/drive/MyDrive/symAD-ECNN').exists():
    PROJECT_ROOT = Path('/content/drive/MyDrive/symAD-ECNN')
else:
    PROJECT_ROOT = REPO_ROOT

# Ensure module paths
EVALS_DIR = REPO_ROOT / 'notebooks' / 'evals'
ECNN_DIR = EVALS_DIR / 'ecnn_thresholding'
for p in [REPO_ROOT, EVALS_DIR, ECNN_DIR]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Install missing python packages
def ensure_pkg(import_name: str, pip_name: str = None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        print(f'Installing {pip_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])

ensure_pkg('skimage', 'scikit-image')
ensure_pkg('e2cnn', 'e2cnn')

from skimage.metrics import structural_similarity as ssim

OUT_DIR = PROJECT_ROOT / 'evaluations' / 'tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'IN_COLAB: {IN_COLAB}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'REPO_ROOT: {REPO_ROOT}')
print(f'Device: {device}')

In [ ]:
# TURBO LOADER (optional): unzip IXI fast zips to local Colab disk
import shutil
from pathlib import Path

data_paths = {}
BASE_DIR = Path('/content/drive/MyDrive/symAD-ECNN/data')
LOCAL_DIR = Path('/content/local_data')

ZIPS_TO_EXTRACT = {
    'train': BASE_DIR / 'train_fast.zip',
    'val':   BASE_DIR / 'val_fast.zip',
    'test':  BASE_DIR / 'test_fast.zip',
}

if Path('/content').exists():
    print('Extracting to Local Disk...')
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)

    for name, zip_path in ZIPS_TO_EXTRACT.items():
        target_dir = LOCAL_DIR / name
        if target_dir.exists():
            shutil.rmtree(target_dir)
        target_dir.mkdir(parents=True, exist_ok=True)

        if zip_path.exists():
            print(f'  Unzipping {name}...')
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(target_dir)
        else:
            print(f'  Skipped (not found): {zip_path}')

    data_paths['ixi_train'] = LOCAL_DIR / 'train'
    data_paths['ixi_val'] = LOCAL_DIR / 'val'
    data_paths['ixi_test'] = LOCAL_DIR / 'test'

    print('\nUpdated data_paths:')
    for key, path in data_paths.items():
        status = 'Found' if path.exists() else 'Not found'
        print(f'  {key}: {status}')
        print(f'    -> {path}')
else:
    print('Not in Colab. Skipping turbo unzip step.')

In [ ]:
# BRATS LOADER (explicit): unzip BraTS processed test set and register path
brats_data_path = None

if Path('/content').exists():
    BRATS_BASE = Path('/content/test_eval_data')
    BRATS_ZIP_CANDIDATES = [
        Path('/content/drive/MyDrive/symAD-ECNN/data/brats_test_with_masks_processed.zip'),
        PROJECT_ROOT / 'data' / 'brats_test_with_masks_processed.zip',
    ]

    brats_zip = next((p for p in BRATS_ZIP_CANDIDATES if p.exists()), None)
    if brats_zip is not None:
        print(f'Found BraTS zip: {brats_zip}')
        BRATS_BASE.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(brats_zip, 'r') as zf:
            zf.extractall(BRATS_BASE)

        # support both /processed and direct layout
        for candidate in [BRATS_BASE / 'processed', BRATS_BASE]:
            if (candidate / 'images').exists() and (candidate / 'labels').exists():
                brats_data_path = candidate
                break

        if brats_data_path is not None:
            print(f'BraTS data ready: {brats_data_path}')
        else:
            print('BraTS unzip done, but images/labels structure not found.')
    else:
        print('BraTS zip not found in expected locations.')
else:
    # Local/Windows workspace fallback
    for candidate in [PROJECT_ROOT / 'data' / 'test_eval_data' / 'processed', PROJECT_ROOT / 'data' / 'test_eval_data']:
        if (candidate / 'images').exists() and (candidate / 'labels').exists():
            brats_data_path = candidate
            break
    print(f'BraTS data path (local): {brats_data_path}')

In [ ]:
# Model definitions + checkpoint discovery
def ensure_e2cnn_installed():
    if importlib.util.find_spec('e2cnn') is not None:
        return
    print('Installing e2cnn...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'e2cnn'])

try:
    ensure_e2cnn_installed()
    from ecnn_model_loader import get_model_for_inference
    HAS_ECNN = True
except Exception as e:
    HAS_ECNN = False
    print(f'ECNN loader unavailable: {e}')

class CNNAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d(2,2),
        )
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(8 * 8 * 256, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 8 * 8 * 256)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 256, 3, 2, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 3, 2, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 3, 2, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 3, 2, 1, 1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 1, 3, 1, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        feats = self.encoder(x)
        b = feats.size(0)
        z = self.fc_encode(self.flatten(feats))
        dec = self.fc_decode(z).view(b, 256, 8, 8)
        return self.decoder(dec)

class LargeCNNAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True), nn.MaxPool2d(2,2),
        )
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(8 * 8 * 512, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 8 * 8 * 512)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 3, 2, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 3, 2, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 3, 2, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 3, 2, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 1, 3, 1, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        feats = self.encoder(x)
        b = feats.size(0)
        z = self.fc_encode(self.flatten(feats))
        dec = self.fc_decode(z).view(b, 512, 8, 8)
        return self.decoder(dec)

class ResNetAutoencoder(nn.Module):
    def __init__(self, finetune_strategy='none'):
        super().__init__()
        resnet = models.resnet18(weights=None)
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        if finetune_strategy == 'none':
            for p in self.encoder.parameters():
                p.requires_grad = False
        elif finetune_strategy == 'partial':
            for i, module in enumerate(self.encoder.children()):
                train = i >= 7
                for p in module.parameters():
                    p.requires_grad = train
        elif finetune_strategy == 'full':
            for p in self.encoder.parameters():
                p.requires_grad = True
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 1, 0), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

def get_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if 'model_state_dict' in ckpt and isinstance(ckpt['model_state_dict'], dict):
            return ckpt['model_state_dict']
        if 'state_dict' in ckpt and isinstance(ckpt['state_dict'], dict):
            return ckpt['state_dict']
    return ckpt

CHECKPOINT_ROOTS = [
    PROJECT_ROOT / 'models' / 'saved_models',
    REPO_ROOT / 'models' / 'saved_models',
    Path('/content/drive/MyDrive/symAD-ECNN/models/saved_models')
]
CHECKPOINT_ROOTS = [p for p in CHECKPOINT_ROOTS if p.exists()]
print('Checkpoint roots:')
for p in CHECKPOINT_ROOTS:
    print('-', p)

MODEL_CONFIGS = [
    {'name': 'ECNN-AE (Optimized)', 'key': 'ecnn_opt', 'type': 'ecnn', 'subdirs': ['ecnn_optimized', '.'], 'patterns': ['ecnn_optimized_best.pth', '*ecnn*best*.pth', '*ecnn*.pth']},
    {'name': 'CNN-AE', 'key': 'cnn_base', 'type': 'cnn_base', 'subdirs': ['cnn_ae', '.'], 'patterns': ['cnn_best.pth', '*cnn*best*.pth', '*cnn*.pth']},
    {'name': 'CNN-AE Augmented', 'key': 'cnn_aug', 'type': 'cnn_base', 'subdirs': ['cnn_ae_augmented', '.'], 'patterns': ['*cnn*aug*best*.pth', '*aug*cnn*.pth']},
    {'name': 'CNN-AE Large', 'key': 'cnn_large', 'type': 'cnn_large', 'subdirs': ['cnn_ae_large', '.'], 'patterns': ['*cnn*large*best*.pth', '*large*cnn*.pth']},
    {'name': 'ResNet-AE (frozen)', 'key': 'resnet_ae', 'type': 'resnet_none', 'subdirs': ['resnet_ae', '.'], 'patterns': ['*resnet*best*.pth', '*resnet*.pth']},
    {'name': 'ResNet-AE (partial FT)', 'key': 'resnet_ft_partial', 'type': 'resnet_partial', 'subdirs': ['resnet_finetuned_partial', 'resnet_finetuned', '.'], 'patterns': ['*resnet*best*.pth', '*resnet*.pth']},
    {'name': 'ResNet-AE (full FT)', 'key': 'resnet_ft_full', 'type': 'resnet_full', 'subdirs': ['resnet_finetuned_full', 'resnet_finetuned', '.'], 'patterns': ['*resnet*best*.pth', '*resnet*.pth']},
]

def find_best_checkpoint(cfg):
    cands = []
    for root in CHECKPOINT_ROOTS:
        for sd in cfg['subdirs']:
            base = root if sd == '.' else root / sd
            if not base.exists():
                continue
            for pat in cfg['patterns']:
                cands.extend([p for p in base.rglob(pat) if p.is_file()])
    cands = sorted(set(cands))
    if not cands:
        return None
    cands = sorted(cands, key=lambda p: (('best' not in p.name.lower()), -p.stat().st_mtime))
    return cands[0]

def load_model_from_cfg(cfg):
    ckpt_path = find_best_checkpoint(cfg)
    if ckpt_path is None:
        raise FileNotFoundError('No checkpoint found')

    if cfg['type'] == 'ecnn':
        if not HAS_ECNN:
            raise RuntimeError('ECNN loader unavailable')
        model, _ = get_model_for_inference(str(ckpt_path), str(device))
        return model.to(device).eval(), ckpt_path

    ckpt = torch.load(ckpt_path, map_location=device)
    sd = get_state_dict(ckpt)

    if cfg['type'] == 'cnn_large':
        latent = int(sd['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in sd else 512
        model = LargeCNNAutoencoder(latent_dim=latent)
    elif cfg['type'] == 'cnn_base':
        latent = int(sd['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in sd else 256
        model = CNNAutoencoder(latent_dim=latent)
    elif cfg['type'] == 'resnet_none':
        model = ResNetAutoencoder(finetune_strategy='none')
    elif cfg['type'] == 'resnet_partial':
        model = ResNetAutoencoder(finetune_strategy='partial')
    elif cfg['type'] == 'resnet_full':
        model = ResNetAutoencoder(finetune_strategy='full')
    else:
        raise ValueError(f"Unknown type: {cfg['type']}")

    model.load_state_dict(sd, strict=False)
    return model.to(device).eval(), ckpt_path

In [ ]:
# Dataset + metric helpers
class EvalDataset(Dataset):
    def __init__(self, image_dir: Path, label_dir: Path, mask_dir=None, mode='grayscale'):
        self.image_files = sorted(image_dir.glob('slice_*.npy'))
        self.label_dir = label_dir
        self.mask_dir = mask_dir if (mask_dir is not None and Path(mask_dir).exists()) else None
        self.mode = mode
        self.resnet_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        sid = img_path.stem.split('_')[-1]
        lab_path = self.label_dir / f'label_{sid}.npy'
        img = np.load(img_path).astype(np.float32)
        label = int(np.load(lab_path)[0])

        gray = torch.from_numpy(img).float().unsqueeze(0)
        target = gray.clone()

        if self.mask_dir is not None:
            mpath = Path(self.mask_dir) / f'slice_{sid}.npy'
            mask = np.load(mpath).astype(np.uint8) if mpath.exists() else np.zeros_like(img, dtype=np.uint8)
        else:
            mask = np.zeros_like(img, dtype=np.uint8)

        if self.mode == 'resnet':
            img_uint8 = (img * 255.0).clip(0, 255).astype(np.uint8)
            rgb = np.stack([img_uint8, img_uint8, img_uint8], axis=-1)
            inp = self.resnet_transform(rgb)
            return inp, target, label, sid, torch.from_numpy(mask).unsqueeze(0).float()

        return gray, target, label, sid, torch.from_numpy(mask).unsqueeze(0).float()

def normalize01(x):
    x = x.astype(np.float32)
    mn, mx = float(x.min()), float(x.max())
    if mx - mn < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn)

def dice_score(pred, gt):
    pred = pred.astype(np.uint8)
    gt = gt.astype(np.uint8)
    inter = (pred * gt).sum()
    return float((2.0 * inter) / (pred.sum() + gt.sum() + 1e-8))

def iou_score(pred, gt):
    pred = pred.astype(np.uint8)
    gt = gt.astype(np.uint8)
    inter = (pred * gt).sum()
    union = ((pred + gt) > 0).sum()
    return float(inter / (union + 1e-8))

def threshold_from_normals(scores, labels, target_fpr=0.20):
    normal_scores = scores[labels == 0]
    if len(normal_scores) == 0:
        return float(np.percentile(scores, 80))
    return float(np.percentile(normal_scores, (1 - target_fpr) * 100))

def compute_cls_metrics(labels, scores, threshold):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)
    preds = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold),
        'auroc': float(roc_auc_score(labels, scores)) if np.unique(labels).size > 1 else np.nan,
        'auprc': float(average_precision_score(labels, scores)) if np.unique(labels).size > 1 else np.nan,
        'accuracy': float(accuracy_score(labels, preds)),
        'precision': float(precision_score(labels, preds, zero_division=0)),
        'recall': float(recall_score(labels, preds, zero_division=0)),
        'specificity': float(tn / (tn + fp)) if (tn + fp) > 0 else np.nan,
        'f1_score': float(f1_score(labels, preds, zero_division=0)),
        'fpr': float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan,
        'fnr': float(fn / (fn + tp)) if (fn + tp) > 0 else np.nan,
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
    }

def _topk_mean_np(arr_2d: np.ndarray, frac: float) -> np.ndarray:
    n = arr_2d.shape[1]
    k = max(1, int(n * frac))
    part = np.partition(arr_2d, n - k, axis=1)[:, n - k:]
    return part.mean(axis=1)

@torch.no_grad()
def infer_scores_and_lesion_metrics(model, dataloader, pixel_percentile=95):
    score_store = {
        'mse_mean': [], 'mae_mean': [],
        'mse_p95': [], 'mse_p99': [],
        'mae_p95': [], 'mae_p99': [],
        'mse_top1pct_mean': [], 'mae_top1pct_mean': [],
    }

    ssim_scores, labels = [], []
    lesion_dice, lesion_iou = [], []
    lesion_count, non_lesion_count = 0, 0

    for x, target, lab, sid, mask in tqdm(dataloader, leave=False):
        x = x.to(device)
        target = target.to(device)
        mask = mask.to(device)

        recon = model(x)
        if recon.shape[-2:] != target.shape[-2:]:
            recon = F.interpolate(recon, size=target.shape[-2:], mode='bilinear', align_corners=False)

        err_abs = torch.abs(recon - target)
        err_sq = (recon - target) ** 2

        err_abs_np = err_abs.detach().cpu().numpy()[:, 0]
        err_sq_np = err_sq.detach().cpu().numpy()[:, 0]
        flat_abs = err_abs_np.reshape(err_abs_np.shape[0], -1)
        flat_sq = err_sq_np.reshape(err_sq_np.shape[0], -1)

        score_store['mse_mean'].extend(flat_sq.mean(axis=1).tolist())
        score_store['mae_mean'].extend(flat_abs.mean(axis=1).tolist())
        score_store['mse_p95'].extend(np.percentile(flat_sq, 95, axis=1).tolist())
        score_store['mse_p99'].extend(np.percentile(flat_sq, 99, axis=1).tolist())
        score_store['mae_p95'].extend(np.percentile(flat_abs, 95, axis=1).tolist())
        score_store['mae_p99'].extend(np.percentile(flat_abs, 99, axis=1).tolist())
        score_store['mse_top1pct_mean'].extend(_topk_mean_np(flat_sq, 0.01).tolist())
        score_store['mae_top1pct_mean'].extend(_topk_mean_np(flat_abs, 0.01).tolist())

        labels.extend(lab.numpy().tolist())

        t_np = target.detach().cpu().numpy()
        r_np = recon.detach().cpu().numpy()
        for b in range(t_np.shape[0]):
            gt_img = t_np[b, 0].astype(np.float32)
            rc_img = r_np[b, 0].astype(np.float32)
            dr = float(max(gt_img.max(), rc_img.max()) - min(gt_img.min(), rc_img.min()))
            dr = dr if dr > 1e-8 else 1.0
            ssim_scores.append(float(ssim(gt_img, rc_img, data_range=dr)))

        err = err_abs.detach().cpu().numpy()
        msk = (mask.detach().cpu().numpy() > 0).astype(np.uint8)
        for b in range(err.shape[0]):
            gt = msk[b, 0]
            if gt.sum() > 0:
                lesion_count += 1
                emap = normalize01(err[b, 0])
                thr = np.percentile(emap, pixel_percentile)
                pred = (emap >= thr).astype(np.uint8)
                lesion_dice.append(dice_score(pred, gt))
                lesion_iou.append(iou_score(pred, gt))
            else:
                non_lesion_count += 1

    labels = np.asarray(labels, dtype=np.int32)
    ssim_scores = np.asarray(ssim_scores, dtype=np.float32)
    score_store = {k: np.asarray(v, dtype=np.float32) for k, v in score_store.items()}
    return {
        'score_store': score_store,
        'mse_scores': score_store['mse_mean'],
        'mae_scores': score_store['mae_mean'],
        'ssim_scores': ssim_scores,
        'labels': labels,
        'lesion_dice': lesion_dice,
        'lesion_iou': lesion_iou,
        'lesion_count': int(lesion_count),
        'non_lesion_count': int(non_lesion_count),
    }

In [ ]:
# Run all-model inference + build TWO master tables with separate roots
import re
import shutil
from pathlib import Path

def has_required_dirs(p: Path) -> bool:
    return (p / 'images').exists() and (p / 'labels').exists()

def resolve_processed_root(base_path: Path):
    """Return a folder that contains images/ + labels/ (supports direct or /processed)."""
    if base_path is None:
        return None
    base_path = Path(base_path)
    candidates = [base_path / 'processed', base_path]
    for c in candidates:
        if c.exists() and has_required_dirs(c):
            return c
    return None

def _sid_from_name(path_obj: Path):
    m = re.search(r'(\d+)$', path_obj.stem)
    return m.group(1) if m else None

def _is_image_npy(p: Path):
    s = str(p).lower().replace('\\', '/')
    n = p.name.lower()
    if not n.endswith('.npy'):
        return False
    if 'label_' in n or '/labels/' in s:
        return False
    if '/masks/' in s or 'mask' in n:
        return False
    return True

def collect_records(root: Path, source_name: str, force_label=None):
    """Collect file records from one dataset root, robust to nested zip layouts."""
    records = []
    if root is None or not Path(root).exists():
        return records

    root = Path(root)

    if force_label is not None:
        # For IXI side, labels may not exist. Use all image npy files recursively.
        img_files = sorted([p for p in root.rglob('*.npy') if _is_image_npy(p)])
        for img_path in img_files:
            sid = _sid_from_name(img_path)
            mask_path = None
            if sid is not None:
                mask_candidates = [m for m in root.rglob(f'slice_{sid}.npy') if ('/masks/' in str(m).lower().replace('\\', '/')) or ('mask' in m.name.lower())]
                if mask_candidates:
                    mask_path = mask_candidates[0]
            records.append({
                'image_path': img_path,
                'label': int(force_label),
                'mask_path': mask_path,
                'source': source_name,
            })
        return records

    # For BraTS side, prefer label files then match images by sid.
    label_files = sorted(root.rglob('label_*.npy'))
    if not label_files:
        # fallback: if labels are not in label_*.npy format, try any npy with 'label'
        label_files = sorted([p for p in root.rglob('*.npy') if 'label' in p.name.lower()])

    for lab_path in label_files:
        sid = _sid_from_name(lab_path)
        if sid is None:
            continue
        try:
            lab = int(np.load(lab_path)[0])
        except Exception:
            continue

        img_candidates = [p for p in root.rglob(f'slice_{sid}.npy') if _is_image_npy(p)]
        if not img_candidates:
            # flexible fallback by sid anywhere in filename
            img_candidates = [p for p in root.rglob('*.npy') if _is_image_npy(p) and sid in p.stem]

        if not img_candidates:
            continue

        img_path = img_candidates[0]
        mask_candidates = [m for m in root.rglob(f'slice_{sid}.npy') if ('/masks/' in str(m).lower().replace('\\', '/')) or ('mask' in m.name.lower())]
        mask_path = mask_candidates[0] if mask_candidates else None

        records.append({
            'image_path': img_path,
            'label': lab,
            'mask_path': mask_path,
            'source': source_name,
        })

    return records

class MultiSourceEvalDataset(Dataset):
    def __init__(self, records, mode='grayscale'):
        self.records = list(records)
        self.mode = mode
        self.resnet_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img = np.load(rec['image_path']).astype(np.float32)
        label = int(rec['label'])

        gray = torch.from_numpy(img).float().unsqueeze(0)
        target = gray.clone()

        if rec['mask_path'] is not None and Path(rec['mask_path']).exists():
            mask = np.load(rec['mask_path']).astype(np.uint8)
        else:
            mask = np.zeros_like(img, dtype=np.uint8)

        sid = rec['image_path'].stem.split('_')[-1]

        if self.mode == 'resnet':
            img_uint8 = (img * 255.0).clip(0, 255).astype(np.uint8)
            rgb = np.stack([img_uint8, img_uint8, img_uint8], axis=-1)
            inp = self.resnet_transform(rgb)
            return inp, target, label, sid, torch.from_numpy(mask).unsqueeze(0).float()

        return gray, target, label, sid, torch.from_numpy(mask).unsqueeze(0).float()

# ---------- Resolve roots ----------
# BraTS-only root (test_eval_data / explicit loader)
BASE_EVAL_DATA_DIR = Path('/content/test_eval_data') if Path('/content').exists() else (PROJECT_ROOT / 'data' / 'test_eval_data')
PROCESSED_ZIP = PROJECT_ROOT / 'data' / 'brats_test_with_masks_processed.zip'

brats_only_root = None
if 'brats_data_path' in globals() and brats_data_path is not None:
    candidate = Path(brats_data_path)
    if candidate.exists():
        brats_only_root = candidate

if brats_only_root is None:
    brats_candidates = [BASE_EVAL_DATA_DIR / 'processed', BASE_EVAL_DATA_DIR]
    brats_only_root = next((p for p in brats_candidates if p.exists()), None)

if brats_only_root is None and PROCESSED_ZIP.exists():
    print(f'Extracting processed zip: {PROCESSED_ZIP}')
    BASE_EVAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PROCESSED_ZIP, 'r') as zf:
        zf.extractall(BASE_EVAL_DATA_DIR)

    post_extract_candidates = [BASE_EVAL_DATA_DIR / 'processed', BASE_EVAL_DATA_DIR] + [d for d in BASE_EVAL_DATA_DIR.iterdir() if d.is_dir()]
    brats_only_root = next((p for p in post_extract_candidates if p.exists()), None)

if brats_only_root is None:
    raise FileNotFoundError('BraTS-only dataset (0/1) not found in test_eval_data.')

# IXI+BraTS mixed roots (from local_data/val and local_data/test)
ixi_mix_root = None
brats_mix_root = None
if 'data_paths' in globals() and isinstance(data_paths, dict):
    # user requested: IXI from local_data/val, BraTS from local_data/test
    if data_paths.get('ixi_val') is not None:
        ixi_mix_root = Path(data_paths['ixi_val'])
    if data_paths.get('ixi_test') is not None:
        brats_mix_root = Path(data_paths['ixi_test'])

# fallback for mix BraTS root if test does not work
if brats_mix_root is None or not brats_mix_root.exists():
    brats_mix_root = brats_only_root

print(f'IXI+BraTS root (IXI side): {ixi_mix_root if ixi_mix_root is not None else "Not found"}')
print(f'IXI+BraTS root (BraTS side): {brats_mix_root if brats_mix_root is not None else "Not found"}')
print(f'BraTS-only root: {brats_only_root}')

# ---------- Build records ----------
ixi_records = collect_records(ixi_mix_root, 'ixi', force_label=0) if ixi_mix_root is not None else []
brats_mix_records = [r for r in collect_records(brats_mix_root, 'brats_mix') if r['label'] in (0, 1)] if brats_mix_root is not None else []
brats_only_records = [r for r in collect_records(brats_only_root, 'brats_only') if r['label'] in (0, 1)]

# If both are empty, try swapped mapping automatically
if len(ixi_records) == 0 and len(brats_mix_records) == 0 and 'data_paths' in globals() and isinstance(data_paths, dict):
    alt_ixi = Path(data_paths['ixi_test']) if data_paths.get('ixi_test') is not None else None
    alt_brats = Path(data_paths['ixi_val']) if data_paths.get('ixi_val') is not None else None
    alt_ixi_records = collect_records(alt_ixi, 'ixi', force_label=0) if alt_ixi is not None else []
    alt_brats_records = [r for r in collect_records(alt_brats, 'brats_mix') if r['label'] in (0, 1)] if alt_brats is not None else []
    if len(alt_ixi_records) > 0 or len(alt_brats_records) > 0:
        print('Detected usable files with swapped local_data mapping; using swapped roots.')
        ixi_mix_root, brats_mix_root = alt_ixi, alt_brats
        ixi_records, brats_mix_records = alt_ixi_records, alt_brats_records

print(f'IXI records (forced label 0): {len(ixi_records)}')
print(f'Mix BraTS records (label 0/1): {len(brats_mix_records)}')
print(f'BraTS-only records (label 0/1): {len(brats_only_records)}')

dataset_scenarios = {}
if len(ixi_records) > 0 and len(brats_mix_records) > 0:
    dataset_scenarios['ixi_plus_brats'] = ixi_records + brats_mix_records
else:
    print('Skipping ixi_plus_brats (missing usable IXI/BraTS files in local_data val/test).')

if len(brats_only_records) > 0:
    dataset_scenarios['brats_0_1_only'] = brats_only_records

if len(dataset_scenarios) == 0:
    raise RuntimeError('No valid dataset scenario available.')

TARGET_FPR = 0.20
BATCH_SIZE = 32
PIXEL_PERCENTILE = 95

results_by_scenario = {}

for scenario_name, records in dataset_scenarios.items():
    print('\n' + '=' * 90)
    print(f'SCENARIO: {scenario_name}')
    print('=' * 90)

    labels_arr = np.asarray([r['label'] for r in records], dtype=np.int32)
    print(f'Total samples: {len(records)} | label0={int((labels_arr==0).sum())} | label1={int((labels_arr==1).sum())}')

    rows = []
    for cfg in MODEL_CONFIGS:
        print(f'\n=== Evaluating {cfg["name"]} ===')
        try:
            model, ckpt_path = load_model_from_cfg(cfg)
            mode = 'resnet' if 'resnet' in cfg['type'] else 'grayscale'

            ds = MultiSourceEvalDataset(records, mode=mode)
            dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

            out = infer_scores_and_lesion_metrics(model, dl, pixel_percentile=PIXEL_PERCENTILE)
            labels = out['labels']
            score_store = out['score_store']
            mse_scores = out['mse_scores']
            mae_scores = out['mae_scores']
            ssim_scores = out['ssim_scores']

            normal_mask = (labels == 0)
            lesion_mask = (labels == 1)

            if np.unique(labels).size < 2:
                print('Warning: only one class found in labels. AUROC/AUPRC may be NaN.')

            candidate_scores = {
                'mse_mean': score_store['mse_mean'],
                'mae_mean': score_store['mae_mean'],
                'mse_p95': score_store['mse_p95'],
                'mse_p99': score_store['mse_p99'],
                'mae_p95': score_store['mae_p95'],
                'mae_p99': score_store['mae_p99'],
                'mse_top1pct_mean': score_store['mse_top1pct_mean'],
                'mae_top1pct_mean': score_store['mae_top1pct_mean'],
                '1_minus_ssim': (1.0 - ssim_scores),
            }

            score_stats = {}
            if np.unique(labels).size > 1:
                for k, v in candidate_scores.items():
                    score_stats[k] = {
                        'auroc': float(roc_auc_score(labels, v)),
                        'auprc': float(average_precision_score(labels, v)),
                        'gap': float(np.mean(v[lesion_mask]) - np.mean(v[normal_mask])) if (normal_mask.any() and lesion_mask.any()) else np.nan,
                    }
            else:
                for k, _ in candidate_scores.items():
                    score_stats[k] = {'auroc': np.nan, 'auprc': np.nan, 'gap': np.nan}

            best_score_name = max(score_stats, key=lambda k: (score_stats[k]['auroc'] if score_stats[k]['auroc'] == score_stats[k]['auroc'] else -1.0))
            selected_scores = candidate_scores[best_score_name]

            thr = threshold_from_normals(selected_scores, labels, TARGET_FPR)
            cls = compute_cls_metrics(labels, selected_scores, thr)

            row = {
                'dataset_scenario': scenario_name,
                'model_name': cfg['name'],
                'checkpoint': str(ckpt_path),
                'selected_score_name': best_score_name,
                'n_samples': int(len(labels)),
                'n_normal_labels': int(normal_mask.sum()),
                'n_lesion_labels': int(lesion_mask.sum()),
                'n_lesion_slices': int(out['lesion_count']),
                'n_non_lesion_slices': int(out['non_lesion_count']),
                'mean_selected_score_normal': float(np.mean(selected_scores[normal_mask])) if normal_mask.any() else np.nan,
                'mean_selected_score_lesion': float(np.mean(selected_scores[lesion_mask])) if lesion_mask.any() else np.nan,
                'selected_score_gap_lesion_minus_normal': score_stats[best_score_name]['gap'],
                'mean_mse_all': float(np.mean(mse_scores)) if len(mse_scores) else np.nan,
                'mean_mse_normal': float(np.mean(mse_scores[normal_mask])) if normal_mask.any() else np.nan,
                'mean_mse_lesion': float(np.mean(mse_scores[lesion_mask])) if lesion_mask.any() else np.nan,
                'mean_mae_all': float(np.mean(mae_scores)) if len(mae_scores) else np.nan,
                'mean_mae_normal': float(np.mean(mae_scores[normal_mask])) if normal_mask.any() else np.nan,
                'mean_mae_lesion': float(np.mean(mae_scores[lesion_mask])) if lesion_mask.any() else np.nan,
                'mean_ssim_all': float(np.mean(ssim_scores)) if len(ssim_scores) else np.nan,
                'mean_ssim_normal': float(np.mean(ssim_scores[normal_mask])) if normal_mask.any() else np.nan,
                'mean_ssim_lesion': float(np.mean(ssim_scores[lesion_mask])) if lesion_mask.any() else np.nan,
                'mean_dice_lesion_only': float(np.mean(out['lesion_dice'])) if len(out['lesion_dice']) else np.nan,
                'mean_iou_lesion_only': float(np.mean(out['lesion_iou'])) if len(out['lesion_iou']) else np.nan,
                'selected_auroc': score_stats[best_score_name]['auroc'],
                'selected_auprc': score_stats[best_score_name]['auprc'],
                **cls,
            }
            rows.append(row)

            print(f"Selected={best_score_name} | AUROC={row['selected_auroc']:.4f} | AUPRC={row['selected_auprc']:.4f}")
            print(f"MSE={row['mean_mse_all']:.6f} MAE={row['mean_mae_all']:.6f} SSIM={row['mean_ssim_all']:.4f}")

        except Exception as e:
            print(f"FAILED: {cfg['name']} -> {e}")

    df_s = pd.DataFrame(rows)
    if df_s.empty:
        print(f'No models evaluated successfully for scenario: {scenario_name}')
        continue

    df_s = df_s.sort_values('selected_auroc', ascending=False, na_position='last').reset_index(drop=True)
    results_by_scenario[scenario_name] = df_s

    print(f'\nTable for scenario: {scenario_name}')
    display(df_s)

    raw_out = OUT_DIR / f'all_models_full_metrics_{scenario_name}.csv'
    rank_out = OUT_DIR / f'all_models_full_metrics_{scenario_name}_ranked.csv'
    df_s.to_csv(raw_out, index=False)
    df_s.assign(rank=np.arange(1, len(df_s) + 1)).to_csv(rank_out, index=False)
    print(f'Saved: {raw_out}')
    print(f'Saved: {rank_out}')

# Backward-compatible handle
if 'brats_0_1_only' in results_by_scenario:
    results_df = results_by_scenario['brats_0_1_only']
elif len(results_by_scenario) > 0:
    results_df = next(iter(results_by_scenario.values()))
else:
    results_df = pd.DataFrame()

In [ ]:
# Publication-style tables for BOTH scenarios
if 'results_by_scenario' in globals() and isinstance(results_by_scenario, dict) and len(results_by_scenario) > 0:
    publication_tables = {}

    for scenario_name, df in results_by_scenario.items():
        if df is None or df.empty:
            continue

        pub_cols = [
            'model_name', 'selected_score_name', 'threshold',
            'selected_auroc', 'selected_auprc',
            'accuracy', 'precision', 'recall', 'specificity', 'f1_score',
            'mean_mse_all', 'mean_mae_all', 'mean_ssim_all',
            'mean_mse_normal', 'mean_mse_lesion',
            'mean_mae_normal', 'mean_mae_lesion',
            'mean_ssim_normal', 'mean_ssim_lesion'
        ]
        pub_cols = [c for c in pub_cols if c in df.columns]
        pub_df = df[pub_cols].copy()
        pub_df.insert(0, 'Rank', np.arange(1, len(pub_df) + 1))

        pub_df = pub_df.rename(columns={
            'model_name': 'Model',
            'selected_score_name': 'Selected Score',
            'threshold': 'Threshold',
            'selected_auroc': 'AUROC',
            'selected_auprc': 'AUPRC',
            'accuracy': 'Accuracy',
            'precision': 'Precision',
            'recall': 'Recall',
            'specificity': 'Specificity',
            'f1_score': 'F1 Score',
            'mean_mse_all': 'MSE (All)',
            'mean_mae_all': 'MAE (All)',
            'mean_ssim_all': 'SSIM (All)',
            'mean_mse_normal': 'MSE (Healthy)',
            'mean_mse_lesion': 'MSE (Tumour)',
            'mean_mae_normal': 'MAE (Healthy)',
            'mean_mae_lesion': 'MAE (Tumour)',
            'mean_ssim_normal': 'SSIM (Healthy)',
            'mean_ssim_lesion': 'SSIM (Tumour)'
        })

        publication_tables[scenario_name] = pub_df

        print('\n' + '=' * 100)
        print(f'PUBLICATION TABLE: {scenario_name}')
        print('=' * 100)
        display(pub_df)

        pub_out = OUT_DIR / f'all_models_full_metrics_publication_{scenario_name}.csv'
        pub_df.to_csv(pub_out, index=False)
        print(f'Saved: {pub_out}')
else:
    print('No scenario results available. Run the previous cell first.')

In [ ]:
# Sanity check: verify IXI+BraTS roots are mapped correctly
from pathlib import Path
import numpy as np
import re

def _count_npy_images(root: Path):
    if root is None or not Path(root).exists():
        return 0
    root = Path(root)
    cnt = 0
    for p in root.rglob('*.npy'):
        s = str(p).lower().replace('\\', '/')
        n = p.name.lower()
        if n.startswith('label_') or '/labels/' in s:
            continue
        if 'mask' in n or '/masks/' in s:
            continue
        cnt += 1
    return cnt

def _count_label_files(root: Path):
    if root is None or not Path(root).exists():
        return 0
    root = Path(root)
    labels = list(root.rglob('label_*.npy'))
    if len(labels) == 0:
        labels = [p for p in root.rglob('*.npy') if 'label' in p.name.lower()]
    return len(labels)

def _label_distribution_from_files(root: Path, max_files=2000):
    if root is None or not Path(root).exists():
        return {'0': 0, '1': 0, 'other': 0, 'sampled': 0}
    root = Path(root)
    labels = list(root.rglob('label_*.npy'))
    if len(labels) == 0:
        labels = [p for p in root.rglob('*.npy') if 'label' in p.name.lower()]
    labels = labels[:max_files]
    d0 = d1 = dother = 0
    for lp in labels:
        try:
            v = int(np.load(lp)[0])
            if v == 0:
                d0 += 1
            elif v == 1:
                d1 += 1
            else:
                dother += 1
        except Exception:
            dother += 1
    return {'0': d0, '1': d1, 'other': dother, 'sampled': len(labels)}

def _peek_files(root: Path, limit=5):
    if root is None or not Path(root).exists():
        return []
    root = Path(root)
    files = []
    for p in root.rglob('*.npy'):
        files.append(str(p))
        if len(files) >= limit:
            break
    return files

print('=' * 100)
print('ROOT MAPPING CHECK')
print('=' * 100)
print(f"ixi_mix_root   : {ixi_mix_root}")
print(f"brats_mix_root : {brats_mix_root}")
print(f"brats_only_root: {brats_only_root}")

for name, root in [('IXI side', ixi_mix_root), ('Mix BraTS side', brats_mix_root), ('BraTS-only side', brats_only_root)]:
    print('\n' + '-' * 100)
    print(name)
    exists = (root is not None and Path(root).exists())
    print(f"exists: {exists}")
    if not exists:
        continue
    img_n = _count_npy_images(root)
    lab_n = _count_label_files(root)
    dist = _label_distribution_from_files(root)
    print(f"image-like npy count : {img_n}")
    print(f"label file count      : {lab_n}")
    print(f"label distribution    : {dist}")
    print('example files:')
    for fp in _peek_files(root, limit=5):
        print('  ', fp)

print('\n' + '=' * 100)
print('RECORD COUNTS USED BY PIPELINE')
print('=' * 100)
print(f"ixi_records       : {len(ixi_records) if 'ixi_records' in globals() else 'N/A'}")
print(f"brats_mix_records : {len(brats_mix_records) if 'brats_mix_records' in globals() else 'N/A'}")
print(f"brats_only_records: {len(brats_only_records) if 'brats_only_records' in globals() else 'N/A'}")
if 'dataset_scenarios' in globals():
    print('scenarios:', list(dataset_scenarios.keys()))
else:
    print('scenarios: N/A')